#  Training Visualization and Monitoring

This notebook visualizes the training progress of the SLM model.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

## 1. Load Training Logs

In [ ]:
# Function to load training logs
def load_training_logs(log_dir='../logs'):
    logs = {}
    
    # Check for different log formats
    if Path(f'{log_dir}/train_losses.pt').exists():
        logs['train_losses'] = torch.load(f'{log_dir}/train_losses.pt')
        logs['val_losses'] = torch.load(f'{log_dir}/val_losses.pt')
        logs['learning_rates'] = torch.load(f'{log_dir}/learning_rates.pt')
    
    # Check for JSON logs
    if Path(f'{log_dir}/training_metrics.json').exists():
        with open(f'{log_dir}/training_metrics.json', 'r') as f:
            logs.update(json.load(f))
    
    return logs

# Load logs
logs = load_training_logs()

if logs:
    print(f"Found training logs with {len(logs)} metrics")
    for key in logs.keys():
        print(f"  - {key}: {len(logs[key]) if isinstance(logs[key], list) else type(logs[key])}")
else:
    print("No training logs found. Please train the model first.")

## 2. Training Curves

In [ ]:
# Plot training and validation loss
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss curves
if 'train_losses' in logs and 'val_losses' in logs:
    train_losses = [l.cpu().detach() if torch.is_tensor(l) else l for l in logs['train_losses']]
    val_losses = [l.cpu().detach() if torch.is_tensor(l) else l for l in logs['val_losses']]
    
    axes[0, 0].plot(train_losses, label='Train Loss', alpha=0.7, linewidth=2)
    axes[0, 0].plot(val_losses, label='Validation Loss', alpha=0.7, linewidth=2)
    axes[0, 0].set_xlabel('Steps')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

# Learning rate
if 'learning_rates' in logs:
    lrs = logs['learning_rates']
    axes[0, 1].plot(lrs)
    axes[0, 1].set_xlabel('Steps')
    axes[0, 1].set_ylabel('Learning Rate')
    axes[0, 1].set_title('Learning Rate Schedule')
    axes[0, 1].grid(True, alpha=0.3)

# Loss difference
if 'train_losses' in logs and 'val_losses' in logs:
    loss_diff = np.array(val_losses) - np.array(train_losses)
    axes[1, 0].plot(loss_diff, color='purple')
    axes[1, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[1, 0].set_xlabel('Steps')
    axes[1, 0].set_ylabel('Loss Difference (Val - Train)')
    axes[1, 0].set_title('Overfitting Indicator')
    axes[1, 0].grid(True, alpha=0.3)

# Distribution of losses
if 'train_losses' in logs and 'val_losses' in logs:
    axes[1, 1].hist(train_losses, bins=20, alpha=0.5, label='Train')
    axes[1, 1].hist(val_losses, bins=20, alpha=0.5, label='Validation')
    axes[1, 1].set_xlabel('Loss')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Loss Distribution')
    axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../logs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Convergence Analysis

In [ ]:
if 'train_losses' in logs:
    # Calculate moving average
    window = 10
    train_ma = np.convolve(train_losses, np.ones(window)/window, mode='valid')
    val_ma = np.convolve(val_losses, np.ones(window)/window, mode='valid')
    
    # Calculate gradient of loss (rate of change)
    train_gradient = np.gradient(train_ma)
    val_gradient = np.gradient(val_ma)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Moving average
    axes[0].plot(train_ma, label='Train (MA)', linewidth=2)
    axes[0].plot(val_ma, label='Validation (MA)', linewidth=2)
    axes[0].set_xlabel('Steps')
    axes[0].set_ylabel('Loss (Moving Average)')
    axes[0].set_title(f'Loss Convergence (Window={window})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Gradient
    axes[1].plot(train_gradient, label='Train', alpha=0.7)
    axes[1].plot(val_gradient, label='Validation', alpha=0.7)
    axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('Steps')
    axes[1].set_ylabel('Rate of Change')
    axes[1].set_title('Loss Gradient (Convergence Speed)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../logs/convergence_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Performance Metrics

In [ ]:
# Calculate key metrics
metrics = {}

if 'train_losses' in logs:
    final_train_loss = train_losses[-1]
    final_val_loss = val_losses[-1]
    min_val_loss = min(val_losses)
    min_val_step = val_losses.index(min_val_loss)
    
    # Calculate improvement
    initial_train_loss = train_losses[0]
    train_improvement = (initial_train_loss - final_train_loss) / initial_train_loss * 100
    
    metrics = {
        'Initial Train Loss': f'{initial_train_loss:.4f}',
        'Final Train Loss': f'{final_train_loss:.4f}',
        'Final Validation Loss': f'{final_val_loss:.4f}',
        'Best Validation Loss': f'{min_val_loss:.4f}',
        'Best Val Step': min_val_step,
        'Train Improvement (%)': f'{train_improvement:.2f}%',
        'Total Steps': len(train_losses),
    }
    
    # Create metrics DataFrame
    metrics_df = pd.DataFrame([metrics])
    metrics_df
    
    # Save metrics
    metrics_df.to_csv('../logs/training_metrics.csv', index=False)
    print("Metrics saved to logs/training_metrics.csv")

## 5. Compare Different Runs

In [ ]:
# Function to load multiple runs
def load_multiple_runs(base_dir='../logs'):
    runs = {}
    for path in Path(base_dir).glob('run_*'):
        if path.is_dir():
            run_name = path.name
            run_data = {}
            
            # Load losses
            for loss_file in path.glob('*_losses.pt'):
                run_data[loss_file.stem] = torch.load(loss_file)
            
            runs[run_name] = run_data
    
    return runs

# Uncomment below if you have multiple runs
# all_runs = load_multiple_runs()
# print(f"Found {len(all_runs)} training runs")

## 6. Save Visualizations

In [ ]:
# Create summary report
def create_training_report(logs, metrics):
    report = []
    report.append("="*60)
    report.append("TRAINING SUMMARY REPORT")
    report.append("="*60)
    report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append("")
    
    for key, value in metrics.items():
        report.append(f"{key}: {value}")
    
    report.append("")
    report.append("="*60)
    
    return "\n".join(report)

# Save report
if metrics:
    report = create_training_report(logs, metrics)
    with open('../logs/training_report.txt', 'w') as f:
        f.write(report)
    print(report)